# What drives the price of a car?

![](https://github.com/anthonyfeaster/UCB-AIML/blob/main/Practical-2/images/kurt.jpeg?raw=1)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary.

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import plotly as px

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score




#Creating a dataframe from the csv
df = pd.read_csv("data/vehicles.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data/vehicles.csv'

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

In [ ]:
#Beginning the Exploratory Data Analysis of the dataset

df.head() # Taking a peak at the data

Looking at the first 5 rows of this dataset, it is apparent that there are multiple columns with NaN values. This will need be cleaned up before conducting analysis

In [ ]:
#Printing Number of Rows and Columns plus Column Names
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
print(f"Column Names: {df.columns.tolist()}")

This dataset contains 426,880 car records and 18 different features. This is an indication of a large dataset and could be suitable for regression modeling.

In [ ]:
#Printing the Datatypes
df.info()

This dataset contains numerical and categorical attributes. It also contains several NaN values mostly in the categorical fields such as cylinders, size and condition. The missing values will be cleaned up during the next phase

In [ ]:
print("====================================")
print("Columns with Null Values")
print("====================================")
print(df.isnull().sum().sort_values(ascending=False))
print("\n====================================")
print("Columns with Null Values - Percentages")
print("====================================")
((df.isnull().sum() / len(df) * 100).map("{:.2f}%".format)).sort_values(ascending=False)

My intial look into the dataset revealed several NaN entries. This command is showing the extent of the null values and the percentage of null values.

In [ ]:
#Looking at the unique values for the cylinder column
df['cylinders'].value_counts(dropna=False)

In [ ]:
display(df['cylinders'].value_counts(dropna=True))

In [ ]:
#Looking at the unique values for the condition column
df['condition'].value_counts(dropna=False)

In [ ]:
#Looking at the unique values for the drive column
df['drive'].value_counts(dropna=False)

In [ ]:
#Looking at the unique values for the paint_color column
df['paint_color'].value_counts(dropna=False)

Replace missing values with unknown

In [ ]:
df['price'].describe()

In [ ]:
sns.histplot(df['price'], bins=50)
plt.title('Price Distribution')
plt.show()

In [ ]:
df['price'].quantile([0.05, 0.25, 0.5, 0.75, 0.95])

In [ ]:
lower = df['price'].quantile(0.05)
upper = df['price'].quantile(0.95)

df_temp = df[(df['price'] > lower) & (df['price'] < upper)]

sns.histplot(df[df['price'] < 100000]['price'], bins=50)
plt.title('Price Distribution (5th-95th Percentile)')
plt.show()

In [ ]:
sns.histplot(df['year'].dropna(), bins=50)
plt.title('Year Distribution')
plt.show()

This chart illustrates that the distribution of most vehicles are from 2000-2020 with very few being older than 2000. This suggest that the dataset contains mostly recent vehicles.

In [ ]:
sns.histplot(df[df['odometer'] < 300000]['odometer'], bins=50)
plt.title('Odometer Distribution')
plt.show()

The odometer distribution is slightly skewed to the right. The milage for most vehicles are between 20,000 and 150,000. Vehicles with 300,000 miles and above were excluded to improve chart visualization

In [ ]:
df_filtered = df[(df['odometer'] < 300000) & (df['price'] < 100000)]

df_sample = df_filtered.sample(n=1000, random_state=42)

sns.scatterplot(data=df_sample, x='odometer', y='price')
plt.title('Odometer vs Price')
plt.show()

Initial scatterplot was too crowded due to overplotting which made it difficult to interpret the data. I took a random sample of the data to improve the visualization. The plot shows a downward trend where the higher the milage, the lower the price of the vehicle. This suggests that the odometer reading plays a part in the price of the vehicle.

In [ ]:
df_price = df[(df['price'] > 500) & (df['price'] < 50000)]

plt.figure(figsize=(10, 6))
sns.boxplot(x='condition', y='price', data=df_price)
plt.xticks(rotation=45)
plt.title('Condition vs Price')
plt.show()

This boxplot shows the distribution of vehicle prices across various condition categories. Vehicles in better condition tended to have higher listed prices in comparison to vehicles in poor condition. This suggests that the condition of the vehicle is an important attribute in determining the price of the vehicle.

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`.

In [ ]:
df = df.drop(columns=['region', 'id', 'VIN', 'size'])

Dropping the 'id' and 'VIN' columns because they are unique values and do not provide any benefit to the analysis. Dropping the 'region' column because it is redundant info thats captured in the state column. Dropping the 'size' column because it contains 70% null values.

In [ ]:
#Remove price outliers
lower = df['price'].quantile(0.01)
upper = df['price'].quantile(0.99)

df = df[(df['price'] > lower) & (df['price'] < upper)]

In [ ]:
upper_odometer = df['odometer'].quantile(0.99)

df = df[df['odometer'] < upper_odometer]

In [ ]:

df.columns.tolist()

In [ ]:
df['cylinders'].unique()

In [ ]:
#Extracting the Numberical Cylinder value and converting it to a float
df['cylinders'] = df['cylinders'].astype(str).str.extract(r'(\d+)').astype(float)

#Replacing the NaN values with the median
df['cylinders'] = df['cylinders'].fillna(df['cylinders'].median())
print(df['cylinders'].unique())
print(df['cylinders'].dtypes)

In [ ]:
#Showing the number count of each cylinder value
display(df['cylinders'].value_counts(dropna=False))

Cleaned the 'cylinders' column by extracting all of the text and converting the remaining values to numerical data. The value count confirms all rows in the 'cylinder' column have been cleaned and are ready for analysis.

In [ ]:
df['condition'] = df['condition'].fillna('unknown')
print(f"Number of NaN Values: {df['condition'].isnull().sum()}")
print()
print(df['condition'].value_counts())

In [ ]:
df['drive'] = df['drive'].fillna('unknown')
print(df['drive'].isnull().sum())
print(df['drive'].value_counts())

In [ ]:
df['paint_color'] = df['paint_color'].fillna('unknown')
print(f'Number of NaN Values: {df['paint_color'].isnull().sum()}')
print()
print(df['paint_color'].value_counts())

In [ ]:
df.isnull().sum()

In [ ]:
#Writing a for loop to fill all missing (NaN) values from designated columns with 'unknown'
for col in ['manufacturer', 'fuel', 'model', 'title_status', 'transmission', 'type']:
    df[col] = df[col].fillna('unknown')
    print(f'Number of NaN Values for {col}: {df[col].isnull().sum()}')
    print()

In [ ]:
#Dropping rows with a missing year
df = df.dropna(subset=['year'])

In [ ]:
#Validating that all NaN values have been taken care of
df.isnull().sum()

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

In [ ]:
X = df.drop(columns=['price'])
y = df['price']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
numberic_features = ['year', 'odometer', 'cylinders']
categorical_features = ['manufacturer', 'model', 'condition', 'fuel', 'title_status', 'transmission', 'drive', 'type', 'paint_color', 'state']



In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numberic_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [ ]:
lr_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
lr_model.fit(X_train, y_train)
lr_model.score(X_test, y_test)

In [ ]:
ridge_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])
ridge_model.fit(X_train, y_train)
ridge_model.score(X_test, y_test)

In [ ]:
#rf_model = Pipeline([
#   ('preprocessor', preprocessor),
#   ('model', RandomForestRegressor(n_estimators = 4, random_state = 42))
#])
#rf_model.fit(X_train, y_train)
#rf_model.score(X_test, y_test)

Explored using a random forest regressor but due to computational limitations with the environment, the model wasnt fully evaluated.

In [ ]:
lr_cv = cross_val_score(lr_model, X_train, y_train, cv=3)
print(f'Linear Regression CV Score: {lr_cv.mean()}')

In [ ]:
ridge_cv = cross_val_score(ridge_model, X_train, y_train, cv=3)
print(f'Ridge Regression CV Score: {ridge_cv.mean()}')

In [ ]:
#rf_cv = cross_val_score(rf_model, X_train, y_train, cv=5)
#print(f'Cross Validation Scores: {rf_cv.mean()}')

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

In [ ]:
lr_pred = lr_model.predict(X_test)
lr_r2 = lr_model.score(X_test, y_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
print(f'LR R2: {lr_r2}')
print(f'LR RMSE: {lr_rmse}')

In [ ]:
ridge_pred = ridge_model.predict(X_test)
ridge_r2 = ridge_model.score(X_test, y_test)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
print(f'Ridge R2: {ridge_r2}')
print(f'Ridge RMSE: {ridge_rmse}')

In [ ]:
#rf_pred = rf_model.predict(X_test)
#rf_r2 = rf_model.score(X_test, y_test)
#rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge Regression'],
    'R2': [lr_r2, ridge_r2],
    'RMSE': [lr_rmse, ridge_rmse]
})
results

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.

This model can be used by a dealership to estimate the value of used cars but keeping in mind that  market demand can also influence the price. Future improvements to the model can include analyzing whether location influences vehicle prices.


